# Transfer and retention: held-out results
Run `train.py`, `evaluate.py`, and `compare.py` first. Each point below is a complete method run. The dashed lines mark the shared initial adapter. A single-seed difference is descriptive, not proof of improvement.

In [ ]:
from pathlib import Path
import json
import matplotlib.pyplot as plt
import pandas as pd

HERE = Path.cwd()
ROOT = HERE
RESULTS = ROOT / 'outputs' / 'transfer_v1'
table = pd.read_csv(RESULTS / 'comparison.csv')
details = json.loads((RESULTS / 'comparison.json').read_text(encoding='utf-8'))
labels = {'initial': 'A-SFT', 'grpo': 'GRPO', 'dr_grpo': 'Dr. GRPO', 'dapo': 'DAPO', 'pact_grpo': 'PACT-GRPO'}
display(table[['seed', 'method', 'hard_accuracy', 'easy_zero_rejection', 'hard_gain_pp', 'easy_zero_change_pp', 'rollout_tokens', 'train_seconds']].round(3))

In [ ]:
for seed, frame in table.groupby('seed'):
    initial = frame.loc[frame.method == 'initial'].iloc[0]
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), constrained_layout=True)
    for _, row in frame.iterrows():
        axes[0].scatter(100 * row.easy_zero_rejection, 100 * row.hard_accuracy, s=90, label=labels[row.method])
        axes[0].annotate(labels[row.method], (100 * row.easy_zero_rejection, 100 * row.hard_accuracy), xytext=(5, 5), textcoords='offset points', fontsize=8)
    axes[0].axvline(100 * initial.easy_zero_rejection, color='gray', ls='--', lw=1)
    axes[0].axhline(100 * initial.hard_accuracy, color='gray', ls='--', lw=1)
    axes[0].set(xlabel='A-test: reject all extraneous roots (%)', ylabel='B-test: exact-set accuracy (%)', title='Adaptation versus retention')
    rl = frame.loc[frame.method != 'initial']
    axes[1].bar([labels[name] for name in rl.method], rl.rollout_tokens / 1000)
    axes[1].set(ylabel='Generated rollout tokens (thousands)', title='RL sampling cost')
    axes[1].tick_params(axis='x', labelrotation=20)
    fig.suptitle(f'Seed {seed}: held-out test')
    plt.show()

In [ ]:
for seed, item in details['per_seed'].items():
    fig, axes = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)
    for method, curve in item['validation_curves'].items():
        x = [point['rollout_tokens'] for point in curve]
        axes[0].plot(x, [100 * point['hard_accuracy'] for point in curve], marker='o', label=labels[method])
        axes[1].plot(x, [100 * point['easy_zero_rejection'] for point in curve], marker='o', label=labels[method])
    axes[0].set(title='B-validation: new family', xlabel='Generated rollout tokens', ylabel='Exact-set accuracy (%)')
    axes[1].set(title='A-validation: old rejection skill', xlabel='Generated rollout tokens', ylabel='Zero-root rejection (%)')
    axes[1].legend(loc='best', fontsize=8)
    fig.suptitle(f'Seed {seed}: validation trajectory')
    plt.show()

In [ ]:
# The retention axis above covers only no-solution cases; inspect all A subgroups.
for seed, frame in table.groupby('seed'):
    ordered = frame.set_index('method').loc[list(labels)]
    names = [labels[method] for method in ordered.index]
    groups = [('A: zero valid roots', 'easy_zero_rejection'), ('A: one valid root', 'easy_one_accuracy'), ('A: two valid roots', 'easy_two_accuracy')]
    fig, ax = plt.subplots(figsize=(10, 4.5), constrained_layout=True)
    for index, (group_name, column) in enumerate(groups):
        positions = [index + (method_index - 2) * 0.16 for method_index in range(len(names))]
        for method_index, (position, value) in enumerate(zip(positions, ordered[column])):
            ax.bar(position, 100 * value, width=0.15, label=names[method_index] if index == 0 else None)
    ax.set_xticks(range(len(groups)), [name for name, _ in groups])
    ax.set(ylabel='Exact-set accuracy (%)', ylim=(0, 105), title=f'Seed {seed}: old-family test by valid-root count')
    ax.legend(ncol=3, fontsize=8)
    plt.show()